# Step 1: Create the Access Connector

## In Azure Portal

```text
Resource Group
└── rg-retail-dlt-dev
```

Create an **Access Connector for Azure Databricks** with the following name:

| **Property** | **Value** |
|--------------|-----------|
| **Resource Type** | Access Connector for Azure Databricks |
| **Name** | `ac-retail-dlt-dev` |

# Step 2: Grant Storage Permissions

Navigate to:

```text
Storage Account
└── Access Control (IAM)
    └── Add Role Assignment
```

Assign the following role to the Access Connector:

| **Role** | **Principal** |
|----------|---------------|
| **Storage Blob Data Contributor** | `ac-retail-dlt-dev` |

# Step 3. Create the Storage Credential
## Open Databricks SQL Editor.

In [0]:
%sql
CREATE STORAGE CREDENTIAL sc_retail_dlt_dev
WITH AZURE_MANAGED_IDENTITY
'/subscriptions/<subscription-id>/resourceGroups/rg-retail-dlt-dev/providers/Microsoft.Databricks/accessConnectors/ac-retail-dlt-dev';

-- Verify-
SHOW STORAGE CREDENTIALS;

# Step 4: Create the External Location

Assume the following storage account:

| **Property** | **Value** |
|--------------|-----------|
| **Storage Account** | `stretaildltdev` |
| **Container** | `retail` |

### Directory Structure

```text
retail
└── input
    ├── customers
    └── orders


In [0]:
%sql
CREATE EXTERNAL LOCATION el_retail_dlt_dev
URL 'abfss://retail@stretaildltdev.dfs.core.windows.net/input'
WITH (
STORAGE CREDENTIAL sc_retail_dlt_dev
);

-- Verify
SHOW EXTERNAL LOCATIONS;

# Step 5. Create Catalog
## Now create a catalog.

In [0]:
%sql
CREATE CATALOG retail_catalog
MANAGED LOCATION
'abfss://retail@stcapstonedproject01.dfs.core.windows.net/input';

# Step 6. Create Schema

In [0]:
%sql
USE CATALOG retail_catalog;
CREATE SCHEMA retail;

-- Verify
SHOW SCHEMAS;

# Step 7. Create Volume
# Volumes are created inside a schema.

In [0]:
%sql
USE CATALOG retail_catalog;
USE SCHEMA retail;

CREATE VOLUME landing_vol;

-- Verfiy
SHOW VOLUMES;

# Step 8: Upload Files to the Volume

## Navigate to

```text
Catalog Explorer
└── Retail Catalog
    └── retail
        └── Volumes
            └── landing_vol
```

## Create the Following Folder Structure

```text
landing_vol
├── customers
├── orders
├── schema
└── checkpoints
```

In [0]:
%sql
LIST '/Volumes/retail_catalog/retail/landing_vol';


## Upload Files

Upload the following files:

| **File** | **Destination Folder** |
|----------|------------------------|
| `customers.csv` | `customers/` |
| `orders.csv` | `orders/` |